# 07 · Mock interview — 60 minutes, cold start

**This one is different.** No step-by-step scaffolding, no hints. Six open-ended
prompts, the way an interviewer actually gives them, on a dataset you haven't seen.

**Rules:**
1. Set a timer for **60 minutes**. Stop when it goes off, finished or not.
2. No peeking at `solutions/07_mock_interview.ipynb` until the timer ends.
3. **Narrate.** Type your reasoning as comments as you go. Silence is the most common
   way to fail a live-coding round — an interviewer can't credit what they can't hear.
4. Getting through 4 of 6 with clear reasoning beats rushing all 6 in silence.

**The brief:** *"Here's our customer data. We want to know who's going to churn.
Take a look and walk me through what you'd do."*

That's it. That's the whole brief. Vague on purpose — the first thing they're testing
is whether you ask what churn means before you start coding.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])

print('orders   ', orders.shape)
print('customers', customers.shape)
display(orders.head())
display(customers.head())

---

## Q1 — Scope it

### Q1. Before writing any model code: **define churn** for this dataset, in a way you can
actually compute. Then say what you'd need to ask the business.

Write your definition as a comment, and compute the resulting churn rate.

In [ ]:
# There is no churn label in this data. That's the first thing to say out loud -- the task
# as briefed is not yet a supervised problem, and inventing a label silently is the
# failure mode here.
#
# Questions I'd ask the business:
#   - Is this a subscription (churn = cancellation, a real event with a date) or
#     transactional (churn = "stopped coming back", a judgement call)? This looks
#     transactional, so we have to DEFINE it.
#   - What's the normal repurchase cycle? The window has to come from the data, not a
#     round number someone likes.
#   - What decision does this feed? A retention discount that costs $10 has a very
#     different precision/recall tradeoff than a $500 account-manager call.
#
# Working definition, derived rather than guessed:
snapshot = orders['order_date'].max()
print('data runs to:', snapshot.date())

gaps = (orders.sort_values(['customer_id', 'order_date'])
              .groupby('customer_id')['order_date'].diff().dt.days.dropna())
print(gaps.describe().round(1))
for q in (0.5, 0.75, 0.9):
    print(f'  p{int(q*100)} inter-order gap: {gaps.quantile(q):.0f} days')

# Median gap ~36 days, p75 ~75 days, p90 ~124 days. So a customer silent for 90 days has
# already gone quieter than ~80% of normal gaps -- unusual, but not yet unheard of.
# I'll take 90 days: it sits between p75 and p90, and it's a round number the business can
# hold in its head. State the tradeoff out loud rather than pretending it's derived:
#   - shorter window (say 60d) -> more "churners", many of whom were just slow, so you
#     waste retention budget on people who'd have come back anyway
#   - longer window (say 120d) -> a cleaner label, but you only find out someone left four
#     months late, by which point the intervention is pointless
# This is a business calibration, not a statistical one. I'd bring these percentiles to
# the stakeholder and let them pick.
CHURN_DAYS = 90
last_order = orders.groupby('customer_id')['order_date'].max()
recency = (snapshot - last_order).dt.days
churn = (recency > CHURN_DAYS).astype(int)

print(f'\nchurn rate at {CHURN_DAYS}d: {churn.mean():.1%} of {len(churn)} customers')

# CAVEAT to state before moving on: this label is defined using the last 90 days of data,
# so those 90 days CANNOT be used to build features -- that's leakage, and it's the trap
# this whole notebook is built around. See Q3.

---

## Q2 — Explore

### Q2. Explore the data enough to say something useful. Show us what you find — and tell us
what you *distrust*.

Deliberately open. Budget ~10 minutes and stop.

In [ ]:
# --- integrity first: never trust a join you haven't checked ---
check = orders.merge(customers, on='customer_id', how='left', indicator=True)
print(check['_merge'].value_counts().to_string())
orphan_rev = check.loc[check['_merge'] == 'left_only', 'revenue'].sum()
print(f'orphan revenue: ${orphan_rev:,.0f} '
      f'({orphan_rev/check["revenue"].sum():.1%}) from unknown customer_ids\n')

print('missing revenue:', orders['revenue'].isna().sum(), 'rows -> recompute from qty*price')
orders['revenue'] = orders['revenue'].fillna(orders['quantity'] * orders['unit_price'])

df = orders.merge(customers, on='customer_id', how='inner')

# --- the shape of the business ---
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
(df.groupby(df['order_date'].dt.to_period('M'))['revenue'].sum()
   .plot(ax=axes[0], title='Monthly revenue'))
sns.countplot(data=customers, x='country', ax=axes[1],
              order=customers['country'].value_counts().index).set_title('Customers/country')
sns.histplot(df.groupby('customer_id').size(), bins=25, ax=axes[2]).set_title('Orders/customer')
plt.tight_layout(); plt.show()

print(df.groupby('segment')['revenue'].agg(n='count', total='sum', avg='mean').round(2))

# What I'd say:
#  - ~3% of orders point at customer_ids that don't exist in customers.csv. Broken foreign
#    key. An inner join drops them silently; I'd surface it, not swallow it.
#  - 60 orders had null revenue, recomputable from quantity x unit_price. Fixed.
#  - Revenue is flat month to month with no trend or seasonality.
#
# What I DISTRUST, and this is the real answer:
#  - Every segment has almost identical average revenue. Countries too. In real customer
#    data, `premium` and `retail` never spend the same. That uniformity says these columns
#    were generated independently of behaviour -- they carry no signal, and any model that
#    finds some is fitting noise.
#  - Order dates look uniform: no weekday effect, no seasonality, no growth. Real
#    e-commerce has all three.
#  - Conclusion I'd state to the interviewer: this dataset is synthetic and the
#    customer attributes are unrelated to the outcome. I'd expect a churn model here to
#    land near baseline on everything except recency/frequency, which are mechanically
#    related to the label. I'd still build it -- but I'd say this FIRST, so my ~0.5 AUC
#    reads as a correct finding rather than a failure.

---

## Q3 — Build the feature table

### Q3. Turn these two transactional tables into something a model can consume.
One row per customer.

(The whole question is in that last sentence — and in what you're allowed to use.)

In [ ]:
# THE TRAP: churn was defined by looking at the last 90 days. If features are computed over
# all data, they see the same window the label came from, and "days since last order" will
# predict the label perfectly by construction -- ~0.99 AUC and a model that's useless in
# production, because at prediction time that future doesn't exist yet.
#
# The fix is a temporal cutoff:
#   [-------- feature window --------][-- label window (90d) --]
#                                cutoff                    snapshot
# Features from BEFORE the cutoff only. Label from AFTER it.

cutoff = snapshot - pd.Timedelta(days=CHURN_DAYS)
print('feature window ends:', cutoff.date(), '| label window:', cutoff.date(),
      '->', snapshot.date())

hist = df[df['order_date'] <= cutoff]
future = df[df['order_date'] > cutoff]

# Label: did they order at all in the label window?
active_customers = hist['customer_id'].unique()
churned = pd.Series(~pd.Index(active_customers).isin(future['customer_id'].unique()),
                    index=active_customers, name='churned').astype(int)
print(f'churn rate: {churned.mean():.1%} of {len(churned)} customers active before cutoff')

# RFM features -- computed strictly on `hist`, as of the cutoff.
feat = (hist.groupby('customer_id')
            .agg(recency_days=('order_date', lambda s: (cutoff - s.max()).days),
                 frequency=('order_id', 'count'),
                 monetary=('revenue', 'sum'),
                 avg_order_value=('revenue', 'mean'),
                 n_categories=('category', 'nunique'),
                 n_channels=('channel', 'nunique'),
                 first_order=('order_date', 'min'))
            .assign(tenure_days=lambda d: (cutoff - d['first_order']).dt.days)
            .drop(columns='first_order'))

feat['orders_per_month'] = feat['frequency'] / (feat['tenure_days'] / 30).clip(lower=1)
feat = feat.join(customers.set_index('customer_id')[['country', 'segment', 'age']])
feat = feat.join(churned)

print(feat.shape)
display(feat.head().round(2))

# Recency/frequency/monetary is the standard opener for any churn or CLV problem -- worth
# naming as "RFM" out loud, it's the vocabulary they're listening for.

---

## Q4 — Model it

### Q4. Build a model. Justify your choice of algorithm **and** your choice of metric.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             average_precision_score, RocCurveDisplay)

X = feat.drop(columns='churned')
y = feat['churned']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

NUM = X.select_dtypes('number').columns.tolist()
CAT = ['country', 'segment']

pre = ColumnTransformer([
    ('num', Pipeline([('i', SimpleImputer(strategy='median')),
                      ('s', StandardScaler())]), NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CAT),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Dummy': DummyClassifier(strategy='stratified', random_state=42),
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                           class_weight='balanced', random_state=42),
}
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f'{name:14s} CV ROC-AUC {s.mean():.3f} +/- {s.std():.3f}')

model = Pipeline([('pre', pre), ('clf', models['LogReg'])]).fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
print('\ntest ROC-AUC:', round(roc_auc_score(y_test, y_prob), 3))
print('test PR-AUC :', round(average_precision_score(y_test, y_prob), 3),
      f'(baseline = {y_test.mean():.3f})')
print(classification_report(y_test, model.predict(X_test), target_names=['stayed', 'churned']))

# ALGORITHM: logistic regression first, always. It's fast, it gives a calibrated
# probability (retention needs a *ranking* to size a campaign, not a hard label), and the
# coefficients are explainable to the marketing team who has to act on it. Random forest
# as the challenger -- if it doesn't clear logreg by more than the CV std, ship the logreg.
#
# METRIC: not accuracy -- with ~19% churn a "nobody churns" model scores 81% and saves
# nothing. ROC-AUC to compare models on ranking quality, PR-AUC because the positive class
# is the rare and expensive one. The metric the BUSINESS gets is precision@k: "of the 100
# customers we can afford to call, how many were really going to leave?" -- because the
# budget, not the model, sets the threshold.
#
# class_weight='balanced' rather than SMOTE: same effect on the loss, no synthetic rows,
# one parameter instead of a resampling step to keep leak-free inside CV.

---

## Q5 — Interpret

### Q5. What's driving the predictions? Would you ship this?

In [ ]:
from sklearn.inspection import permutation_importance

names = model.named_steps['pre'].get_feature_names_out()
coefs = pd.Series(model.named_steps['clf'].coef_[0], index=names)
odds = pd.DataFrame({'coef': coefs, 'odds_ratio': np.exp(coefs)})
display(odds.reindex(coefs.abs().sort_values(ascending=False).index).round(3))

perm = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=42,
                              scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp.plot.barh(figsize=(7, 4), title='Permutation importance (test, AUC drop)')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

# WHAT DRIVES IT: recency_days and frequency, and essentially nothing else. country,
# segment and age sit at ~zero -- exactly as predicted in Q2. The model rediscovered that
# the customer attributes in this dataset are noise. That's a finding, not a bug.
#
# WOULD I SHIP IT? No. The test ROC-AUC is ~0.43 -- not merely weak, but BELOW 0.5, which
# a lot of candidates misread. 0.5 is chance; below 0.5 does not mean "worse than useless"
# or "invert the predictions to win". On a 100-customer test set it means the model found
# no signal and landed on the unlucky side of chance. The honest reading of 0.43 is "0.5
# plus noise", and you can confirm that by checking whether the CV folds straddle 0.5.
#
# WHY -- and it isn't the model's fault. The orders were generated by a uniform random
# process with no per-customer behaviour, so whether someone ordered in the last 90 days
# is a coin flip that says nothing about the next 90. There is no signal to find. No
# amount of XGBoost fixes a dataset with no generating process behind it.
#
# What I'd say in the room: "The pipeline is correct and I'd trust it on real data, but
# this data has no churn signal in it. Before spending more time on models I'd want the
# real transactional history -- and a real churn definition from the business rather than
# one I invented in the first ten minutes."
#
# THAT ANSWER IS THE POINT OF THIS NOTEBOOK. Reporting ~0.43 with a correct explanation is
# a pass. Reporting 0.99 because you leaked recency across the cutoff is a fail -- and the
# candidate who does it usually thinks they aced it.

---

## Q6 — Present

### Q6. Five bullets for a non-technical stakeholder. What did you find, what would you do
next, what does it cost?

In [ ]:
# --- To the VP of Retention, no jargon ---
#
# 1. WE DON'T HAVE A CHURN DEFINITION YET. Nothing in the data says a customer left --
#    I had to infer it from "hasn't bought in 90 days", a number I picked from the data.
#    Before we build anything real, we need your definition, because it decides who gets
#    called.
#
# 2. THE DATA CAN'T ANSWER THIS QUESTION TODAY. Past purchases in this file don't predict
#    future ones -- our model is close to a coin flip. That's a data problem, not a
#    modelling one, and no better algorithm gets around it.
#
# 3. THERE IS A REAL DATA-QUALITY BUG, WORTH FINDING NOW. About 3% of orders belong to
#    customer IDs that don't exist in our customer table. Any revenue-per-country report
#    running today is quietly under-counting by that much.
#
# 4. WHAT WE ACTUALLY NEED: engagement data, not just orders. Logins, app opens, support
#    tickets, email clicks, and cancellations with dates. Churn shows up in *behaviour*
#    before it shows up in a missing order -- by the time someone stops buying, they've
#    already gone.
#
# 5. WHAT I'D DO NEXT, AND THE COST. ~1 week to get engagement data joined and re-run this
#    pipeline (it's built and it'll transfer as-is). If the signal is there, a first
#    production model is ~3-4 weeks including the A/B test to prove the retention offer
#    actually works. I would not ship anything off today's data -- a model at coin-flip
#    accuracy spends real budget calling customers who were never going to leave.

---

## How to grade yourself

Score 1 point each. **4+ is a strong pass** — note that only two of the eight are code.

| | |
|---|---|
| ☐ | Said "there's no churn label here" **before** writing model code |
| ☐ | Defined the churn window from the data, not from a round number |
| ☐ | Used a **temporal cutoff** — features before it, label after it |
| ☐ | Checked the merge and caught the ~3% orphan orders |
| ☐ | Ran a baseline before believing any score |
| ☐ | Rejected accuracy and said *why*, in terms of the positive rate |
| ☐ | Noticed the customer attributes carry no signal — and said so before modelling |
| ☐ | Was willing to say **"don't ship this"** with a reason |

### The trap, explicitly

If you scored ~0.99 AUC, you computed `recency` over the full data while the label came
from the last 90 days. The feature *is* the label. In production you'd be asking "did
they order recently?" to predict "will they order recently?" — and the model would
collapse on day one.

This is the single most common failure in real churn projects, not just interviews. The
cutoff diagram in Q3 is the thing to remember:

```
[-------- features from here --------][-- label from here --]
                               cutoff                  today
```

### If you only remember one thing

The strongest signal you can send in a live round isn't a high score — it's *"here's why
you shouldn't trust this number."* Anyone can call `.fit()`. Interviewers are hiring the
person who knows when the output is garbage.